# Daily Practice — 2026-10-27 — ML Modelling: Is Your Model Comparison Even Reproducible?

**Dataset:** [Wine Recognition](https://scikit-learn.org/stable/datasets/toy_dataset.html#wine-recognition-dataset) — 178 real chemical analyses of wines grown in the same Italian region by three different cultivars, 13 numeric features (alcohol, malic acid, flavanoids, color intensity, proline, etc.), loaded via `sklearn.datasets.load_wine` (a real analytical-chemistry dataset, not a synthetic one).

## Problem statement

A teammate ran one `train_test_split(random_state=0)`, trained a `RandomForestClassifier` and a `LogisticRegression`, and reported: "Random Forest wins — 97.2% vs 94.4% accuracy, ship Random Forest." That claim rests on a single 80/20 split with a single seed. As the SQA person on this team, you know that a single measurement with no error bars is not evidence of anything — it's one sample from a noisy process. Your job is to build a **repeated-CV comparison harness** that treats "which model is better" as a statistical claim that needs to survive resampling, not a one-shot leaderboard entry.

This is the same discipline as deciding whether a flaky test actually regressed: you don't trust one red run, and you shouldn't trust one green "winner" either.

## What you should produce

1. A data-loading/prep function that returns `X` (as a DataFrame) and `y` for the Wine dataset, with features standardized (`StandardScaler`) since Logistic Regression and SVC are scale-sensitive.
2. `evaluate_model_repeated(model, X, y, n_splits=5, n_repeats=20, random_state=0)` — using `RepeatedStratifiedKFold`, returns a NumPy array of per-fold accuracy scores (length `n_splits * n_repeats`), via `cross_val_score`.
3. `compare_models(scores_a, scores_b, name_a, name_b)` — given two arrays of per-fold scores, computes the mean difference, a 95% bootstrap confidence interval for that difference (resample fold-score pairs with replacement, e.g. 5,000 resamples), and returns a dict with `mean_diff`, `ci_low`, `ci_high`, and `significant` (`True` only if the CI excludes 0).
4. Run the harness for `LogisticRegression(max_iter=5000)`, `RandomForestClassifier(random_state=0)`, and `SVC()` (all with `random_state=0` where applicable). Produce a boxplot of the per-fold accuracy distributions for all three models on one chart.
5. A short write-up (3–5 sentences): does the original "Random Forest beats Logistic Regression by 2.8 points" claim survive repeated CV? What does this imply about trusting a single-split benchmark before merging a "model upgrade" PR, and what would you put in a PR template to prevent this kind of claim from going unchecked?

Try it yourself in the starter cells below before you scroll down to the solution.


## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

RANDOM_STATE = 0


## Load and inspect the data

`load_wine` returns 178 samples across 3 cultivar classes (`class_0`, `class_1`, `class_2`), with
13 numeric chemical features on very different scales (e.g. `proline` is in the hundreds, `nonflavanoid_phenols`
is well under 1). That scale mismatch is exactly why unscaled Logistic Regression / SVC comparisons
against a scale-invariant Random Forest can be misleading — make sure your prep step standardizes
features before any linear/SVM model sees them.

In [ ]:
data = load_wine()
X_raw = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="cultivar")

print(X_raw.shape)
print(y.value_counts().sort_index())
X_raw.describe().T[["mean", "std", "min", "max"]]


## Your task

Fill in the TODOs below. Signatures are given — don't change them, the solution cells assume these names.

In [ ]:
def load_and_prep_wine():
    """TODO: load the wine dataset and return (X, y) where X is a DataFrame
    of the 13 features scaled with StandardScaler (fit on the full X — this
    harness only does CV comparison, not a held-out deployment split), and
    y is the integer cultivar label Series. Keep the original column names
    on the scaled DataFrame.
    """
    raise NotImplementedError


X, y = None, None


In [ ]:
def evaluate_model_repeated(model, X, y, n_splits=5, n_repeats=20, random_state=RANDOM_STATE):
    """TODO: use RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats,
    random_state=random_state) with cross_val_score(model, X, y, cv=cv,
    scoring='accuracy') and return the resulting array of per-fold scores
    (length n_splits * n_repeats).
    """
    raise NotImplementedError


In [ ]:
def compare_models(scores_a, scores_b, name_a, name_b, n_boot=5000, random_state=RANDOM_STATE):
    """TODO: compute mean_diff = mean(scores_a) - mean(scores_b). Then build
    a 95% bootstrap CI for that difference: n_boot times, resample len(scores_a)
    indices with replacement independently for each array (np.random.default_rng),
    recompute the mean difference, collect it, and take the 2.5th/97.5th
    percentiles as ci_low/ci_high. Return a dict:
    {'name_a': name_a, 'name_b': name_b, 'mean_diff': ..., 'ci_low': ...,
     'ci_high': ..., 'significant': bool} where significant is True only if
    the interval [ci_low, ci_high] does not contain 0.
    """
    raise NotImplementedError


In [ ]:
# TODO:
# 1. Prep the data with load_and_prep_wine().
# 2. Run evaluate_model_repeated for LogisticRegression(max_iter=5000),
#    RandomForestClassifier(random_state=RANDOM_STATE), and SVC()
#    (set random_state=RANDOM_STATE on every model that accepts it),
#    storing each model's per-fold score array in a dict keyed by model name.
# 3. Print each model's mean +/- std accuracy across all folds.

X, y = None, None
model_scores = {}


In [ ]:
# TODO:
# 1. Draw a boxplot (matplotlib) with one box per model showing the
#    distribution of per-fold accuracy scores from model_scores.
# 2. Call compare_models for the Random-Forest-vs-LogisticRegression pair
#    (the original claim in dispute) and print the result dict.
# 3. Also compare Random Forest vs SVC, and Logistic Regression vs SVC,
#    printing both results.


### Write-up

TODO: in 3–5 sentences, state whether "Random Forest beats Logistic Regression by 2.8 points"
survives repeated CV, what the bootstrap CI on the difference tells you, and what you'd add to a
PR template to stop an unverified single-split benchmark claim from merging.

*(your answer here)*

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
# 1. Data prep
def load_and_prep_wine():
    data = load_wine()
    X_raw = pd.DataFrame(data.data, columns=data.feature_names)
    y = pd.Series(data.target, name="cultivar")
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=X_raw.columns)
    return X_scaled, y

X, y = load_and_prep_wine()

# 2. Repeated-CV harness
def evaluate_model_repeated(model, X, y, n_splits=5, n_repeats=20, random_state=RANDOM_STATE):
    cv = RepeatedStratifiedKFold(
        n_splits=n_splits, n_repeats=n_repeats, random_state=random_state
    )
    return cross_val_score(model, X, y, cv=cv, scoring="accuracy")

# 3. Bootstrap comparison of two score arrays
def compare_models(scores_a, scores_b, name_a, name_b, n_boot=5000, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    mean_diff = scores_a.mean() - scores_b.mean()
    boot_diffs = np.empty(n_boot)
    n_a, n_b = len(scores_a), len(scores_b)
    for i in range(n_boot):
        resample_a = scores_a[rng.integers(0, n_a, n_a)]
        resample_b = scores_b[rng.integers(0, n_b, n_b)]
        boot_diffs[i] = resample_a.mean() - resample_b.mean()
    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
    return {
        "name_a": name_a,
        "name_b": name_b,
        "mean_diff": mean_diff,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "significant": not (ci_low <= 0 <= ci_high),
    }

# 4. Run all three models
models = {
    "LogisticRegression": LogisticRegression(max_iter=5000),
    "RandomForest": RandomForestClassifier(random_state=RANDOM_STATE),
    "SVC": SVC(random_state=RANDOM_STATE) if "random_state" in SVC().get_params() else SVC(),
}

model_scores = {
    name: evaluate_model_repeated(model, X, y) for name, model in models.items()
}

for name, scores in model_scores.items():
    print(f"{name}: {scores.mean():.4f} +/- {scores.std():.4f} (n={len(scores)} folds)")

# 5. Boxplot
plt.figure(figsize=(7, 5))
plt.boxplot(
    [model_scores[name] for name in models],
    labels=list(models.keys()),
)
plt.ylabel("Per-fold accuracy")
plt.title("Repeated 5-fold CV (20 repeats) accuracy distribution by model")
plt.show()

# 6. Pairwise comparisons
rf_vs_lr = compare_models(
    model_scores["RandomForest"], model_scores["LogisticRegression"],
    "RandomForest", "LogisticRegression",
)
rf_vs_svc = compare_models(
    model_scores["RandomForest"], model_scores["SVC"], "RandomForest", "SVC",
)
lr_vs_svc = compare_models(
    model_scores["LogisticRegression"], model_scores["SVC"], "LogisticRegression", "SVC",
)
print(rf_vs_lr)
print(rf_vs_svc)
print(lr_vs_svc)
```

**Explanation**

Across 100 folds (5 splits × 20 repeats), all three models typically land within a point or two of
each other — commonly in the 96–99% range — because the Wine dataset's three cultivars are nearly
linearly separable with scaled features. The single-split claim ("97.2% vs 94.4%, a 2.8-point win
for Random Forest") usually does **not** survive: the bootstrap 95% CI on `mean(RF) - mean(LR)`
typically straddles zero, meaning the apparent gap is well within the noise produced by *which rows
happened to land in the test fold*, not a real difference in the models' generalization ability. The
teammate's original split was simply an unlucky (or lucky) draw for one of the two models.

The practical lesson: a "model B beats model A" claim backed by one `train_test_split` call is not
evidence — it's an anecdote with a precise-looking decimal point. A PR template for a model-upgrade
change should require (1) the comparison metric computed via repeated/stratified CV rather than a
single split, (2) an explicit uncertainty estimate (std across folds, or a CI on the paired
difference) alongside the point estimate, and (3) a stated significance threshold the improvement
must clear before "upgrade" is allowed to merge — otherwise you're approving noise as a feature.

</details>